In [1]:
import pandas as pd
import json

In [2]:
df = pd.read_csv("../Datasets/triplets.csv")

In [3]:
relations = df["relacion"].unique().tolist()
relations.remove("P10607")
relations.remove("P2643")

In [4]:
with open("relations.txt", "w", encoding="utf-8") as f:
    for r in relations:
        f.write(f"{r}\n")

In [5]:
df[df["relacion"] == "P10607"]

,entidad,relacion,objeto,source_file
6975,Universidad Americana,P10607,American University Eagles,chile
29040,Instituto de Tecnología de Georgia,P10607,Georgia Tech Yellow Jackets,el_salvador
68726,Universidad Estatal de San José,P10607,San Jose State Spartans,usa


In [6]:
nacimiento = df[df["relacion"] == "Lugar de nacimiento"]
nacimiento

,entidad,relacion,objeto,source_file
0,Ariel Fernández,Lugar de nacimiento,Bahía Blanca,argentina
10,Paula Ormaechea,Lugar de nacimiento,Sunchales,argentina
14,Mercedes Paz,Lugar de nacimiento,San Miguel de Tucumán,argentina
18,Elisa Carrió,Lugar de nacimiento,Resistencia,argentina
25,José Evaristo de Uriburu,Lugar de nacimiento,Salta,argentina
...,...,...,...,...
74633,Juan Röhl,Lugar de nacimiento,Caracas,venezuela
74637,Yessica Maria Paz Hidalgo,Lugar de nacimiento,Maracay,venezuela
74641,Giovanni Carrara,Lugar de nacimiento,El Tigre,venezuela
74645,Giovanni Frigo,Lugar de nacimiento,Venezuela,venezuela


In [7]:
def merge_text(row):
    start = "Contesta la siguiente pregunta: ¿Dónde nació "
    end = "?"
    return start + row["entidad"] + end

In [8]:
result = nacimiento.apply(merge_text, axis=1).reset_index()

In [9]:
result.tail()[0].tolist()

['Contesta la siguiente pregunta: ¿Dónde nació Juan Röhl?',
 'Contesta la siguiente pregunta: ¿Dónde nació Yessica Maria Paz Hidalgo?',
 'Contesta la siguiente pregunta: ¿Dónde nació Giovanni Carrara?',
 'Contesta la siguiente pregunta: ¿Dónde nació Giovanni Frigo?',
 'Contesta la siguiente pregunta: ¿Dónde nació Giovanni Romero?']

In [10]:
with open("to_QA.json", encoding="utf-8") as f:
    QA = json.load(f)

In [11]:
global_start = QA.get("start")

In [12]:
def fila_a_QA(row):
    relation = row["relacion"]
    template = QA.get(relation)

    if not template:
        return None        

    start_rel = template.get("start")
    end_rel = template.get("end")

    if relation == "santo patrón" and row["entidad"][0].islower():
        start_rel = "¿Quién es el santo patrón del "
        end_rel = "?"

    question = f"{global_start}{start_rel}{row['entidad']}{end_rel}"
    answer = row["objeto"]

    return pd.Series({
        "entidad": row["entidad"],
        "relacion": row["relacion"],
        "objeto": row["objeto"],
        "obtenido_de": row["source_file"],
        "pregunta": question,
        "respuesta": answer
    })

In [13]:
qa = df.apply(fila_a_QA, axis=1).dropna().reset_index(drop=True)

In [14]:
qa.head(20)["pregunta"].tolist()

['Contesta la siguiente pregunta: ¿Dónde nació Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Cuál es la ciudadanía de Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Dónde estudió Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Quién es el empleador de Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Qué premio recibió Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Cuál es la fecha de nacimiento de Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Quién es el empleador de Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Quién es el empleador de Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Quién es el empleador de Ariel Fernández?',
 'Contesta la siguiente pregunta: ¿Dónde nació Paula Ormaechea?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Paula Ormaechea?',
 'Contesta la siguiente pregunta: ¿Cuál es la ciudadanía de Paula Ormaechea?',
 'Contest

In [15]:
qa.head()

,entidad,relacion,objeto,obtenido_de,pregunta,respuesta
0,Ariel Fernández,Lugar de nacimiento,Bahía Blanca,argentina,Contesta la siguiente pregunta: ¿Dónde nació A...,Bahía Blanca
1,Ariel Fernández,Ocupación,biofísico,argentina,Contesta la siguiente pregunta: ¿Cuál es la oc...,biofísico
2,Ariel Fernández,Ciudadanía,Argentina,argentina,Contesta la siguiente pregunta: ¿Cuál es la ci...,Argentina
3,Ariel Fernández,Educado en,Universidad Nacional del Sur,argentina,Contesta la siguiente pregunta: ¿Dónde estudió...,Universidad Nacional del Sur
4,Ariel Fernández,Empleador,Universidad de Chicago,argentina,Contesta la siguiente pregunta: ¿Quién es el e...,Universidad de Chicago


In [16]:
qa["pregunta"] = qa["pregunta"].str.strip()
qa["respuesta"] = qa["respuesta"].str.strip()

qa_compact = (
    qa.groupby("pregunta", as_index=False)
      .agg(
          entidad=("entidad", "first"),
          relacion=("relacion", "first"),
          respuestas=("respuesta", lambda s: [r for r in s]),
          objetos=("objeto", lambda s: [o for o in s]),
          obtenido_de=("obtenido_de", lambda s: [x for x in s])
      )
)

In [17]:
qa_compact.head()

,pregunta,entidad,relacion,respuestas,objetos,obtenido_de
0,Contesta la siguiente pregunta: ¿A quién o a q...,Departamento de Ahuachapán,reemplaza a,[Partido de Ahuachapán],[Partido de Ahuachapán],[el_salvador]
1,Contesta la siguiente pregunta: ¿A quién o a q...,Director General de Correos,reemplaza a,[Administrador General de Correos],[Administrador General de Correos],[el_salvador]
2,Contesta la siguiente pregunta: ¿A quién o a q...,León,reemplaza a,[Ruínas de León Viejo],[Ruínas de León Viejo],[nicaragua]
3,Contesta la siguiente pregunta: ¿A quién o a q...,Orden de Klement Gottwald por la Construcción ...,reemplaza a,[Orden de la Construcción de la Patria Sociali...,[Orden de la Construcción de la Patria Sociali...,[chile]
4,Contesta la siguiente pregunta: ¿A quién o a q...,Universidad de Salamanca,reemplaza a,[Estudio General de Salamanca],[Estudio General de Salamanca],[guatemala]


In [18]:
qa_compact[qa_compact["relacion"] == "Ocupación"]

,pregunta,entidad,relacion,respuestas,objetos,obtenido_de
20251,Contesta la siguiente pregunta: ¿Cuál es la oc...,"""Weird Al"" Yankovic",Ocupación,"[actor, comediante, cantautor]","[actor, comediante, cantautor]","[usa, usa, usa]"
20252,Contesta la siguiente pregunta: ¿Cuál es la oc...,Aaliyah,Ocupación,"[bailarín de ballet, fotomodelo, modelo, baila...","[bailarín de ballet, fotomodelo, modelo, baila...","[usa, usa, usa, usa, usa, usa, usa]"
20253,Contesta la siguiente pregunta: ¿Cuál es la oc...,Aarón Bardales,Ocupación,[futbolista],[futbolista],[honduras]
20254,Contesta la siguiente pregunta: ¿Cuál es la oc...,Aarón Galindo,Ocupación,[futbolista],[futbolista],[mexico]
20255,Contesta la siguiente pregunta: ¿Cuál es la oc...,Aarón Padilla Gutiérrez,Ocupación,[futbolista],[futbolista],[mexico]
...,...,...,...,...,...,...
29222,Contesta la siguiente pregunta: ¿Cuál es la oc...,Óscar Vargas,Ocupación,[futbolista],[futbolista],[honduras]
29223,Contesta la siguiente pregunta: ¿Cuál es la oc...,Óscar Varona,Ocupación,[baloncestista],[baloncestista],[cuba]
29224,Contesta la siguiente pregunta: ¿Cuál es la oc...,Óscar Wirth,Ocupación,[futbolista],[futbolista],[chile]
29225,Contesta la siguiente pregunta: ¿Cuál es la oc...,Óscar Zepeda,Ocupación,[futbolista],[futbolista],[honduras]


In [19]:
qa_compact[qa_compact["relacion"] == "Ocupación"]["pregunta"].tolist()

['Contesta la siguiente pregunta: ¿Cuál es la ocupación de "Weird Al" Yankovic?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Aaliyah?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Aarón Bardales?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Aarón Galindo?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Aarón Padilla Gutiérrez?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Abdalá Bucaram?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Abdiel J. Adames?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Abdul Vas?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Abdón Calderón Garaycoa?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Abel Alves?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Abel Balbo?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Abel Barrera Hernández?',
 'Contesta la siguiente pregunta: ¿Cuál es la ocupación de Abel 

In [23]:
qa_compact = qa_compact[["entidad", "relacion", "objetos", "pregunta", "respuestas", "obtenido_de"]]

In [24]:
qa_compact.head(2)

,entidad,relacion,objetos,pregunta,respuestas,obtenido_de
0,Departamento de Ahuachapán,reemplaza a,[Partido de Ahuachapán],Contesta la siguiente pregunta: ¿A quién o a q...,[Partido de Ahuachapán],[el_salvador]
1,Director General de Correos,reemplaza a,[Administrador General de Correos],Contesta la siguiente pregunta: ¿A quién o a q...,[Administrador General de Correos],[el_salvador]


In [25]:
qa_compact.to_csv("QA_dataset.csv", index=False)

Revisar:
- categoría para las personas que nacieron aquí
- categoría principal del tema
- lista de interés para el proyecto Wikimedia (muy pocas tripletas y sin sentido)
- subdividido en (división administrativa) (muy pocas tripletas y con muchas respuestas correctas que no tenemos)
- categoría de miembros
- opuesto a (hay tripletas sin sentido "embajador de Colombia,opuesto a,embajador en Colombia")
- descrito en la fuente (es ambiguo)
- forma jurídica (casi ninguna tripleta)
- categoría para las personas que estudiaron en esta institución
- categoría para empleados de la organización
- designado por (ambiguo)
- calendario académico (todas las tripletas tienen la misma respuesta)
- editorial (tiene significados distintos dependiendo de la entidad)
- sujeto tiene rol (ambiguo)
- elemento operado (ambiguo)
- evento significativo (muchas respuestas correctas posibles que no tenemos)
- ubicación (tripletas ambiguas "centrocampista,ubicación,campo de fútbol")
- forma artística (ambiguo)
- categoría para los galardonados con este premio
- sucedido por (ambiguo)
- operador (ambiguo)
- área de operación (muy pocas tripletas)
- diferente de (ambiguo: Florida,diferente de,Florida)
- rango inmediatamente inferior: ambiguo y pocas tripletas
- rango inmediatamente superior: pocas tripletas
- organización dirigida desde este cargo: ambiguo
- categoría que contiene subcategorías epónimas
- se dice que es lo mismo que: (ambiguo: "comandante,se dice que es lo mismo que,comandante")
- vestimenta: ambiguo
- categoría relacionada
- edición o traducción: (ambiguo, "Las fuerzas extrañas,edición o traducción,Las Fuerzas Extrañas")
- archivado en: (ambiguo)
- discografía: (ambiguo, "Ministry,discografía,discografía de Ministry")
- distribución: ambiguo
- caracterizado por: pregunta demasiado abierta
- precedido por (ambiguo)
- faceta de (ambiguo)
- lista del elemento (ambiguo)
- ocupación (con minúscula, ambiguo)
- cargo ocupado por el jefe de gobierno (ambiguo)
- texto regulador principal (ambiguo)
- objetivo del proyecto o de la acción (respuestas muy amplias)
- secretario general (muy pocas tripletas)
- influenciado por (respuestas muy amplias)
- ubicación narrativa (ambiguo, pocas tripletas)
- industria (todas las entidades con el mismo objeto)
- creador (ambiguo)
- escudo de armas (ambiguo)
- propietario de (ambiguo)
- propiedad de (mismo caso que antes)
- título nobiliario (pocas tripletas)
- premio recibido (con minúscula, ambiguo)
- califica para el evento (ambiguo)
- mantenido por el wikiproyecto (demasiado específico)
- confiere (ambiguo)
- categoría para las personas que fallecieron aquí (ambiguo)
- reemplaza a (ambiguo)
- afiliación (respuestas amplias)
- serie (respuestas amplias)
- bandera (ambiguo "Esmeraldas,bandera,Bandera de Esmeraldas")
- color (ambiguo)
- geografía (ambiguo)
- lista de (ambiguo)
- estatus patrimonial (ambiguo)
- historia (ambiguo)
- cultura (ambiguo)
- teléfono de emergencia (pocas tripletas)
- cantidad física medida (ambiguo)
- organizador (ambiguo)
- movimiento (pocas tripletas)
- colección (ambiguo)
- lugar de sepultura (pocas tripletas)
- sexo o género (se superpone con género)
- país de nacionalidad (ambiguo, se superpone con ciudadanía)
- nombre de pila (pocas posibilidades)
- parcialmente coincidente con (ambiguo)
- situado en la entidad geográfica (pocas tripletas, se superpone con otras)
- formato de periódico (muy específica)
- manifestación de (ambiguo, "activista por el clima,manifestación de,activismo climático")
- estadio (pocas tripletas, muy especifica)
- liga (pocas tripletas, muy especifica)
- empleador (se superpone con Empleador)
- arquitecto (muy específico)
- condado histórico (ambiguo)
- color oficial (muy específico)
- representa a (muy específico)
- basado en (respuestas muy amplias)
- libretista (se superpone con guionista)
- estatus de los derechos de autor (muy específico)
- instancia tiene partes de la clase (muy específico)
- presentador (muy específico)
- suplente del cargo (muy específico)
- utiliza (muy específico)
- descripción del sello (muy específico)
- reemplazado por (ambiguo)
- distrito escolar (ambiguo)
- portal principal del tema (muy específico)
- banco central (muy específico)
- empresa productora (se superpone, muy específico)
- idioma de la película o programa de televisión (muy específico, se superpone)
- descubridor o inventor (muy específico, se superpone)
- fabricante (muy específico, se superpone)
- ubicado en la entidad territorial administrativa actual (muy específico)
- cargo ocupado por el dirigente (muy específico)
- prefijo honorífico (muy específico)
- punto más alto (muy específico)
- signatario (muy específico)
- clasificación CNC (Francia) (muy específico jajaja)
- examen realizado (muy específico)
- calificación Medierådet (muy específico)
- lugar de ambientación (muy específico)
- director ejecutivo (se superpone)
- clima (pocas tripletas)
- del universo narrativo (muy específico)
- presente en la obra (muy específico)
- orden religiosa (muy específico)
- situado en la calle (muy específico)
- núcleo de población (muy específico)
- instrumentación (muy específico)
- uso (muy específico)
- característica de (muy específico)
- editor (muy específico)
- inspirado por (respuestas muy amplias)
- mantenido por (muy específico)
- donado por (muy específico)
- sistema de escritura (muy específico)
- órgano legislativo (muy específico)
- alimentación (muy específico)
- desembocadura del curso de agua (muy específico)
- afluente (muy específico)
- cuenca hidrográfica (muy específico)
- punto más bajo (muy específico)
- lado de conducción (muy específico)
- categoría para las películas rodadas aquí
- situado en la entidad territorial estadística (muy específico)
- lista de monumentos (muy específico)
- Estatuto de observador oficial en la organización (muy específico)
- tipo de enchufe eléctrico (muy específico)
- día del año de una ocurrencia periódica (muy específico)
- diócesis (muy específico)
- álbum de banda sonora (muy específico)
- emisora original (muy específico)
- personajes (muy específico)
- moneda (muy específico)
- tiene obras en la colección (muy específico)
- participó en el conflicto (muy específico)
- contiene la entidad territorial estadística (muy específico)
- estilo arquitectónico (muy específico)
- categoría para los mapas de este elemento